<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [75]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torchvision.ops import box_iou
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [76]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

Создаем датасет для предобработки данных

In [77]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()
        boxes = [[x, y, x + w, y + h] for x, y, w, h in boxes]

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [78]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.RandomResizedCrop(size=(512, 512), scale=(0.8, 1.0), p=0.5),
        A.HorizontalFlip(p=0.5),
        A.Affine(translate_percent=(0.05, 0.05), scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.5),
        ], p=0.7),
        A.OneOf([
            A.GaussNoise(p=0.3),
            A.CoarseDropout(p=0.3),
            A.MotionBlur(blur_limit=3, p=0.2),
            A.MedianBlur(blur_limit=3, p=0.2),
        ], p=0.5),
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    # Раскомментируй, если аугментации изменяют ббоксы.
    # Не забудь указать верный формат для ббоксов.
    bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

Не забываем инициализировать наш датасет

In [79]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [80]:
class Backbone(nn.Module):
    def __init__(self, model, unfreeze_last=0):
        super().__init__()
        self.model = model
        for param in self.model.parameters():
            param.requires_grad = False
        layers = list(self.model.children())
        for layer in layers[-unfreeze_last:]:
            for param in layer.parameters():
                param.requires_grad = True

    def forward(self, x):
        return self.model(x)

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [81]:
class Neck(nn.Module):
    def __init__(self, in_channels_list, out_channels):
        super().__init__()
        self.lateral_convs = nn.ModuleList()
        self.output_convs = nn.ModuleList()
        for in_channels in in_channels_list:
            self.lateral_convs.append(nn.Conv2d(in_channels, out_channels, kernel_size=1))
            self.output_convs.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))

    def forward(self, features):
        last_inner = self.lateral_convs[-1](features[-1])
        outputs = [self.output_convs[-1](last_inner)]
        for idx in range(len(features) - 2, -1, -1):
            lateral_feat = self.lateral_convs[idx](features[idx])
            last_inner = F.interpolate(last_inner, size=lateral_feat.shape[-2:], mode='nearest') + lateral_feat
            outputs.insert(0, self.output_convs[idx](last_inner))
        return outputs

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [82]:
class Head(nn.Module):
    def __init__(self, in_channels_list, out_channels):
        super().__init__()
        self.lateral_convs = nn.ModuleList()
        self.output_convs = nn.ModuleList()
        for in_channels in in_channels_list:
            self.lateral_convs.append(nn.Conv2d(in_channels, out_channels, kernel_size=1))
            self.output_convs.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))

    def forward(self, features):
        last_inner = self.lateral_convs[-1](features[-1])
        outputs = [self.output_convs[-1](last_inner)]
        for idx in range(len(features) - 2, -1, -1):
            lateral_feat = self.lateral_convs[idx](features[idx])
            last_inner = F.interpolate(last_inner, size=lateral_feat.shape[-2:], mode='nearest') + lateral_feat
            outputs.insert(0, self.output_convs[idx](last_inner))
        return outputs

Теперь можно снова реализовать класс детектора с учетом всех частей выше!



In [83]:
from torchvision.ops import nms

class Detector(nn.Module):
    def __init__(self, backbone, in_channels_list, fpn_out_channels, num_classes, unfreeze_last=0):
        super().__init__()
        self.backbone = Backbone(backbone, unfreeze_last=unfreeze_last)
        self.neck = Neck(in_channels_list, fpn_out_channels)
        self.num_classes = num_classes
        self.cls_heads = nn.ModuleList([nn.Conv2d(fpn_out_channels, num_classes, 3, padding=1) for _ in in_channels_list])
        self.box_heads = nn.ModuleList([nn.Conv2d(fpn_out_channels, 4, 3, padding=1) for _ in in_channels_list])

    def forward(self, x, score_threshold=0.1, nms_threshold=0.5):
        features = self.backbone(x)
        if isinstance(features, dict):
            features = list(features.values())
        fpn_outs = self.neck(features)

        batch_size = x.size(0)
        all_boxes, all_scores, all_logits, all_labels = [], [], [], []
        for _ in range(batch_size):
            all_boxes.append([]); all_scores.append([])
            all_logits.append([]); all_labels.append([])

        for fmap, cls_head, box_head in zip(fpn_outs, self.cls_heads, self.box_heads):
            cls_logits = cls_head(fmap)
            cls_scores = torch.sigmoid(cls_logits)
            box_pred = box_head(fmap)

            B, C, H, W = cls_scores.shape
            cls_scores = cls_scores.permute(0, 2, 3, 1).reshape(B, -1, C)
            box_pred = box_pred.permute(0, 2, 3, 1).reshape(B, -1, 4)

            for b in range(B):
                max_s, labs = cls_scores[b].max(dim=1)
                all_boxes[b].append(box_pred[b])
                all_scores[b].append(max_s)
                all_logits[b].append(cls_logits.permute(0,2,3,1).reshape(B,-1,C)[b].max(dim=1)[0])
                all_labels[b].append(labs)

        return [
            {
                'boxes': torch.cat(all_boxes[b]),
                'scores': torch.cat(all_scores[b]),
                'logits': torch.cat(all_logits[b]),
                'labels': torch.cat(all_labels[b])
            } for b in range(batch_size)
        ]

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [84]:
def TAL_assigner(pred_boxes, pred_scores, gt_boxes, topk=5, alpha=6.0, beta=1.0):
    num_pred = pred_boxes.shape[0]
    num_gt = gt_boxes.shape[0]
    assigned_gt_idx = torch.full((num_pred,), -1, dtype=torch.long, device=pred_boxes.device)
    if num_gt == 0 or num_pred == 0:
        return assigned_gt_idx

    ious = box_iou(pred_boxes, gt_boxes)
    scores = pred_scores.unsqueeze(1).repeat(1, num_gt)
    t = scores.pow(alpha) * ious.pow(beta)

    px = (pred_boxes[:,0] + pred_boxes[:,2]) / 2
    py = (pred_boxes[:,1] + pred_boxes[:,3]) / 2
    is_in_gt = (px[:, None] >= gt_boxes[None,:,0]) & (px[:, None] <= gt_boxes[None,:,2]) & \
               (py[:, None] >= gt_boxes[None,:,1]) & (py[:, None] <= gt_boxes[None,:,3])
    t = t * is_in_gt.float()

    for gt_idx in range(num_gt):
        _, topk_idx = t[:, gt_idx].topk(min(topk, num_pred))
        for idx in topk_idx:
            if assigned_gt_idx[idx] == -1 or ious[idx, gt_idx] > ious[idx, assigned_gt_idx[idx]]:
                assigned_gt_idx[idx] = gt_idx

    return assigned_gt_idx

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [85]:
from torchvision.ops import distance_box_iou_loss

In [86]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [87]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)
pred_boxes = pred_boxes.float()
true_boxes = true_boxes.float()

In [88]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.0187084674835205


In [89]:
def diou_loss(pred_boxes, gt_boxes):
    px1, py1, px2, py2 = pred_boxes[:,0], pred_boxes[:,1], pred_boxes[:,2], pred_boxes[:,3]
    gx1, gy1, gx2, gy2 = gt_boxes[:,0], gt_boxes[:,1], gt_boxes[:,2], gt_boxes[:,3]

    ix1 = torch.max(px1, gx1)
    iy1 = torch.max(py1, gy1)
    ix2 = torch.min(px2, gx2)
    iy2 = torch.min(py2, gy2)

    iw = (ix2 - ix1).clamp(min=0)
    ih = (iy2 - iy1).clamp(min=0)
    inter = iw * ih

    area_p = (px2 - px1) * (py2 - py1)
    area_g = (gx2 - gx1) * (gy2 - gy1)
    union = area_p + area_g - inter
    iou = inter / union

    cx_p = (px1 + px2) / 2
    cy_p = (py1 + py2) / 2
    cx_g = (gx1 + gx2) / 2
    cy_g = (gy1 + gy2) / 2
    center_dist2 = (cx_p - cx_g)**2 + (cy_p - cy_g)**2

    xc1 = torch.min(px1, gx1)
    yc1 = torch.min(py1, gy1)
    xc2 = torch.max(px2, gx2)
    yc2 = torch.max(py2, gy2)
    c_diag2 = (xc2 - xc1)**2 + (yc2 - yc1)**2

    diou = 1 - iou + center_dist2 / (c_diag2 + 1e-7)
    return diou.mean()

In [90]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

Добиться хоть какого-то малейшего адекватного mAP оказалось выше моих сил...

In [91]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    collate_fn=lambda x: tuple(zip(*x))
)

val_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=4,
    collate_fn=lambda x: tuple(zip(*x))
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [92]:
from tqdm import tqdm
device = 'cuda' if torch.cuda.is_available() else 'cpu'
import torchvision.models as models
from torchvision.models._utils import IntermediateLayerGetter


backbone_cnn = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
backbone = IntermediateLayerGetter(backbone_cnn, return_layers={
    'layer2': '0',
    'layer3': '1',
    'layer4': '2'
})


in_channels_list = [512, 1024, 2048]
fpn_out_channels = 256
num_classes = 4

model = Detector(backbone, in_channels_list, fpn_out_channels, num_classes, unfreeze_last=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [98]:
def train_one_epoch(model, dataloader, optimizer, device, epoch=0):
    model.train()
    total_loss = 0.0

    for imgs, targets in tqdm(dataloader, desc="Training"):
        imgs = torch.stack(imgs).to(device)
        targets = [
            {k: (v.to(device) if isinstance(v, torch.Tensor) else torch.tensor(v, device=device))
             for k, v in t.items()}
            for t in targets
        ]

        optimizer.zero_grad()
        preds = model(imgs)

        losses = []

        for pred, target in zip(preds, targets):
            pred_boxes = pred['boxes']
            pred_logits = pred['logits']
            gt_boxes = target['boxes']

            if pred_boxes.numel() == 0 or gt_boxes.numel() == 0:
                continue

            with torch.no_grad():
                pred_scores = torch.sigmoid(pred_logits)

            if epoch < 2:
                ious = box_iou(pred_boxes, gt_boxes)
                assigned_idx = ious.argmax(dim=1)
                mask = ious.max(dim=1).values >= 0.3
                assigned_idx[~mask] = -1
            else:
                assigned_idx = TAL_assigner(pred_boxes, pred_scores, gt_boxes, topk=7)
                mask = assigned_idx >= 0

            if mask.sum() > 0:
                b_loss = diou_loss(pred_boxes[mask], gt_boxes[assigned_idx[mask]])
            else:
                b_loss = torch.tensor(0.0, device=device)

            cls_target = torch.zeros_like(pred_logits)
            cls_target[mask] = 1.0
            c_loss = F.binary_cross_entropy_with_logits(pred_logits, cls_target, reduction='mean')

            loss = b_loss + 0.5 * c_loss
            losses.append(loss)

        if len(losses) > 0:
            batch_loss = sum(losses)
            batch_loss.backward()
            optimizer.step()
            total_loss += batch_loss.item()
        else:
            optimizer.zero_grad()

    return total_loss / len(dataloader)

In [94]:
!pip install torchmetrics

In [95]:
from torchmetrics.detection import MeanAveragePrecision
@torch.no_grad()
def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.01, nms_threshold=0.5, **kwargs):
    model.eval()
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")

    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = torch.stack(images).to(device)

        processed_targets = []
        for t in targets:
            t_dict = {k: (v.to(device) if isinstance(v, torch.Tensor) else torch.tensor(v, device=device)) for k, v in t.items()}
            if 'labels' not in t_dict:
                t_dict['labels'] = torch.ones(t_dict['boxes'].shape[0], device=device, dtype=torch.int64)
            processed_targets.append(t_dict)

        outputs = model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)

        metric.update(predicts, processed_targets)
        return metric.compute()["map"].item()

In [96]:
def filter_predictions(preds, score_threshold=0.1, nms_threshold=0.5):
    filtered = []
    for p in preds:
        boxes = p['boxes']
        scores = p['scores']
        labels = p.get('labels', torch.ones(scores.shape[0], device=boxes.device, dtype=torch.int64))
        keep = scores >= score_threshold
        boxes = boxes[keep]
        scores = scores[keep]
        labels = labels[keep]
        if boxes.numel() == 0:
            filtered.append({'boxes': torch.zeros((0,4), device=boxes.device),
                             'scores': torch.zeros((0,), device=boxes.device),
                             'labels': torch.zeros((0,), device=boxes.device, dtype=torch.int64)})
            continue
        keep_idx = nms(boxes, scores, nms_threshold)
        boxes = boxes[keep_idx]
        scores = scores[keep_idx]
        labels = labels[keep_idx]
        filtered.append({'boxes': boxes, 'scores': scores, 'labels': labels})
    return filtered

In [99]:
import io

best_map = 0.0
for epoch in range(10):
    train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch=epoch)
    map_score = validate(dataloader=val_loader, filter_predictions_func=filter_predictions,
                         device=device, box_format="xyxy", score_threshold=0.01, nms_threshold=0.5)
    if map_score > best_map:
        best_map = map_score
        torch.save(model.state_dict(), "best_detector.pth")
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Validation mAP: {map_score:.4f}")

Training: 100%|██████████| 116/116 [00:20<00:00,  5.58it/s]


Epoch 1 | Train Loss: 0.0288 | Validation mAP: 0.0000


Training: 100%|██████████| 116/116 [00:21<00:00,  5.37it/s]


Epoch 2 | Train Loss: 0.0000 | Validation mAP: 0.0000


Training: 100%|██████████| 116/116 [00:24<00:00,  4.77it/s]


...

Epoch 8 | Train Loss: 2.9973 | Validation mAP: 0.0000


Training: 100%|██████████| 116/116 [00:24<00:00,  4.65it/s]


Epoch 9 | Train Loss: 2.9554 | Validation mAP: 0.0000


Training: 100%|██████████| 116/116 [00:24<00:00,  4.79it/s]
                                                          

Epoch 10 | Train Loss: 2.9359 | Validation mAP: 0.0000


Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [ ]:
# from torchmetrics.detection import MeanAveragePrecision

# @torch.no_grad()
# def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
#     """ Метод для валидации модели.
#     Возвращает mAP (0.5 ... 0.95).
#     """
#     self.model.eval()
#     # Считаем метрику mAP с помощью функции из torchmetrics
#     metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
#     for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
#         images = images.to(device)
#         outputs = self.model(images)
#         predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
#         metric.update(predicts, targets)
#     return metric.compute()["map"].item()
